![Henry Logo](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)

# Agent2Agent (A2A): Análisis de Influencias Literarias Cervantinas

---

**Curso:** AI Engineering | **Módulo:** Bases de Datos Vectoriales

---

## Objetivos de Aprendizaje

Al finalizar este notebook, serás capaz de:

1. **Explicar** el patrón Agent2Agent (A2A) con handoffs reales del OpenAI Agents SDK.
2. **Distinguir** handoffs de llamadas a tools: cuándo usar cada uno.
3. **Diseñar** una cadena de agentes especializados con responsabilidades distintas.
4. **Implementar** el patrón con `handoff()` del SDK y verificar la traza en OpenAI.
5. **Evaluar** cuantitativamente Simple RAG vs sistema A2A en cobertura temática.

---

### Contexto: Por qué este caso de uso

El análisis literario profundo requiere **síntesis de dos dominios distintos**:
- **Texto primario** (el Quijote): evidencia textual concreta
- **Crítica literaria** (papers académicos): interpretaciones y conexiones

Un solo agente generalista mezcla ambos dominios y pierde profundidad. Con A2A:
- Un **archivista** recupera evidencia textual y crítica
- Un **analista** identifica motivos y patrones
- Un **sintetizador** produce el análisis final

Esto es A2A: agentes especializados que se pasan el trabajo con handoffs.

---

### Comparación con notebooks anteriores

| Notebook | Patrón | SDK |
|----------|--------|-----|
| `02_rag_tfidf.ipynb` | Single Agent RAG | OpenAI Agents SDK |
| `05_rags_vectorial_databases.ipynb` | Clases Python custom | Sin SDK real |
| **`06_agent2agent_literario.ipynb`** | **A2A con handoffs reales** | **OpenAI Agents SDK** |


## Reproducibilidad

```bash
cd 02-vector_data_bases
uv sync                    # instala openai-agents, nest-asyncio y demás deps
cp ../.env .env            # o configurar OPENAI_API_KEY directamente
```

**Dependencias clave:** `openai-agents>=0.0.9`, `nest-asyncio>=1.6.0`, `chromadb`, `langchain-openai`, `tiktoken`.

> Este notebook es **independiente** de `02_rag_tfidf.ipynb`. No comparte variables ni estado.


In [ ]:
# ==============================================================================
# SETUP: Importaciones y configuracion del entorno
# ==============================================================================

from __future__ import annotations

import asyncio
import json
import time
import urllib.request

import chromadb
import matplotlib.pyplot as plt
import nest_asyncio
import numpy as np
import pandas as pd
import seaborn as sns
import tiktoken
from agents import Agent, Runner, function_tool, handoff, trace
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field

# nest_asyncio permite usar asyncio.run() dentro de Jupyter
nest_asyncio.apply()

# --- Configuracion visual ---
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11

# --- Cargar API key ---
load_dotenv(find_dotenv())

# --- Modelos ---
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
enc = tiktoken.encoding_for_model("gpt-4o-mini")

print("Setup completo. Modelos inicializados.")

## Por qué dos corpus

El análisis literario profundo exige dos fuentes de conocimiento distintas:

| Corpus | Contenido | Para qué sirve |
|--------|-----------|---------------|
| **A — Quijote** | ~15K palabras del texto original | Evidencia textual concreta, citas, episodios |
| **B — Crítica literaria** | 5 documentos académicos sintéticos | Interpretaciones, conexiones con otros autores, marco teórico |

Un agente que solo tiene el Quijote puede citar el texto pero no conectar con García Márquez o Borges.  
Un agente que solo tiene crítica literaria puede teorizar pero sin evidencia textual concreta.

La arquitectura A2A permite que el **ArchivistaLiterario** consulte ambos corpus y pase la evidencia al **AnalistaTemasLiterarios**, quien la procesa y la pasa al **SintetizadorLiterario**.


In [ ]:
# ==============================================================================
# CORPUS A: Descargar Quijote de Project Gutenberg, chunking, ChromaDB
# ==============================================================================

QUIJOTE_URL = "https://www.gutenberg.org/cache/epub/2000/pg2000.txt"
QUIJOTE_PATH = "/tmp/quijote_a2a.txt"

print("Descargando el Quijote de Project Gutenberg...")
urllib.request.urlretrieve(QUIJOTE_URL, QUIJOTE_PATH)

with open(QUIJOTE_PATH, encoding="utf-8", errors="replace") as f:
    raw_text = f.read()

# Extraer el cuerpo principal (saltar encabezado/pie de Gutenberg)
start_marker = "PRIMERA PARTE"
end_marker = "End of Project Gutenberg"
start_idx = raw_text.find(start_marker)
end_idx = raw_text.find(end_marker)
if start_idx == -1:
    start_idx = 0
if end_idx == -1:
    end_idx = len(raw_text)
body = raw_text[start_idx:end_idx]

# Chunking: 800 chars / 120 overlap (consistente con notebook 01)
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
quijote_chunks = splitter.split_text(body)

print(f"Texto extraído: {len(body):,} caracteres")
print(f"Chunks generados: {len(quijote_chunks)}")

# --- Construir índice ChromaDB para el Quijote ---
chroma_client_a2a = chromadb.Client()
try:
    chroma_client_a2a.delete_collection("quijote_a2a")
except Exception:
    pass
col_quijote = chroma_client_a2a.create_collection("quijote_a2a")

BATCH_SIZE = 50
for i in range(0, len(quijote_chunks), BATCH_SIZE):
    batch = quijote_chunks[i : i + BATCH_SIZE]
    embeds = embeddings_model.embed_documents(batch)
    col_quijote.add(
        documents=batch,
        embeddings=embeds,
        ids=[f"q_{i+j}" for j in range(len(batch))],
    )

print(f"\nChromaDB 'quijote_a2a': {col_quijote.count()} chunks indexados")

In [ ]:
# ==============================================================================
# CORPUS B: Critica literaria sintetica (5 documentos)
# ==============================================================================

# Documentos académicos sintéticos que representan el corpus de crítica literaria.
# En producción serían PDFs reales; aquí los definimos directamente en el notebook.

CRITICA_DOCS = [
    {
        "id": "critica_001",
        "titulo": "La locura como metáfora epistémica en Cervantes",
        "texto": (
            "La locura de Don Quijote no es una condición médica sino una metáfora epistemológica. "
            "Cervantes anticipa el escepticismo cartesiano al confrontar al caballero con una realidad "
            "que no coincide con su percepción. El molino de viento es el topos más citado: ¿qué es "
            "real, el gigante que ve Quijote o el molino que ve Sancho? Descartes preguntará medio siglo "
            "después: ¿cómo distinguir el sueño de la vigilia? Cervantes ya lo dramatizó. La locura "
            "quijotesca es el precio de la coherencia interna: el caballero mantiene un sistema "
            "cognitivo consistente en sí mismo, aunque colisione con el consenso social. Esto lo conecta "
            "con el constructivismo epistemológico del siglo XX: la realidad como construcción."
        ),
    },
    {
        "id": "critica_002",
        "titulo": "Idealismo quijotesco y el modernismo hispanoamericano",
        "texto": (
            "La influencia del Quijote en la literatura hispanoamericana del siglo XX es profunda y "
            "documentada. García Márquez reconoció a Cervantes como su mayor influencia: el realismo "
            "mágico es, en esencia, el quijotismo aplicado al contexto latinoamericano. Los personajes "
            "de Macondo coexisten con lo sobrenatural con la misma naturalidad que Don Quijote con sus "
            "gigantes. Borges fue más explícito: en 'Pierre Menard, autor del Quijote' deconstruye la "
            "autoría y la originalidad usando el texto cervantino como vehículo. Rulfo, en Pedro Páramo, "
            "lleva el idealismo quijotesco al extremo: Juan Preciado busca a su padre en un pueblo de "
            "muertos, un viaje de quijotesca imposibilidad. El idealismo como motor narrativo es la "
            "herencia más duradera de Cervantes en Hispanoamérica."
        ),
    },
    {
        "id": "critica_003",
        "titulo": "Sancho Panza: racionalismo popular y contrapeso epistémico",
        "texto": (
            "Sancho Panza encarna el empirismo popular frente al racionalismo libresco de su amo. "
            "La dialéctica Quijote-Sancho es la dialéctica idealista-empirista: uno ve gigantes, el "
            "otro molinos; uno ve castillos, el otro ventas. Sin embargo, la relación se complica: "
            "Sancho también se quijotiza progresivamente. En la segunda parte acepta la gobernación "
            "de la Ínsula Barataria con la misma seriedad con que Quijote acepta sus caballerías. "
            "El contrapeso epistémico de Sancho no es simple negación sino diálogo: el realismo "
            "popular que conoce los límites del mundo pero anhela superarlos. Esto convierte al par "
            "en el primer dúo filosófico de la literatura occidental moderna."
        ),
    },
    {
        "id": "critica_004",
        "titulo": "El Quijote como primer ejemplo de metaficción occidental",
        "texto": (
            "La segunda parte del Quijote (1615) es el primer ejercicio sistemático de metaficción "
            "en la literatura occidental. Los personajes saben que son personajes: han leído la "
            "primera parte y discuten su propia representación. Cervantes rompe el cuarto muro tres "
            "siglos antes de que el concepto existiera. Esta técnica influenció directamente a Borges "
            "(cuyos laberintos textuales son metaficción pura), a Nabokov (Pnin, Lolita con sus "
            "narradores poco fiables) y a la novela posmoderna en general. El narrador Cide Hamete "
            "Benengeli, fuente árabe ficticia del texto, introduce la duda sobre la veracidad de toda "
            "narración. Calvino, Eco, y DFW son herederos directos de esta tradición cervantina."
        ),
    },
    {
        "id": "critica_005",
        "titulo": "Rocinante y el simbolismo del medio de transporte heroico",
        "texto": (
            "Rocinante es la antítesis del caballo heroico clásico. Donde Bucéfalo simboliza la "
            "fuerza de Alejandro y Babieca el vigor del Cid, Rocinante es un jamelgo decrépito cuyo "
            "nombre mismo ('rocín antes') subraya su mediocridad pretérita. Este anticlímax deliberado "
            "es clave en la parodia cervantina: el heroísmo quijotesco se afirma a pesar del vehículo, "
            "no gracias a él. En la literatura del absurdo del siglo XX este patrón reaparece: el "
            "vagabundo de Beckett sin destino, el Gregor Samsa de Kafka transformado en insecto. "
            "El héroe degradado que mantiene su misión es una herencia directa del modelo cervantino. "
            "Rocinante es el primer antihéroe vehicular de la tradición literaria occidental."
        ),
    },
]

# --- Construir índice ChromaDB para la crítica literaria ---
try:
    chroma_client_a2a.delete_collection("critica_literaria_a2a")
except Exception:
    pass
col_critica = chroma_client_a2a.create_collection("critica_literaria_a2a")

critica_textos = [d["texto"] for d in CRITICA_DOCS]
critica_embeds = embeddings_model.embed_documents(critica_textos)
col_critica.add(
    documents=critica_textos,
    embeddings=critica_embeds,
    ids=[d["id"] for d in CRITICA_DOCS],
    metadatas=[{"titulo": d["titulo"]} for d in CRITICA_DOCS],
)

print("Corpus B - Crítica literaria indexada:")
for d in CRITICA_DOCS:
    print(f"  [{d['id']}] {d['titulo']}")
print(f"\nChromaDB 'critica_literaria_a2a': {col_critica.count()} documentos")

## Modelos Pydantic y Function Tools

Los **Pydantic models** definen el contrato de datos entre agentes.  
Los **function_tools** son las capacidades concretas de cada agente — acceso a ChromaDB.

### Por qué separar las tools por corpus

- `buscar_texto_quijote()` → solo accede al corpus A (texto primario)
- `buscar_critica_literaria()` → solo accede al corpus B (crítica académica)

Esto permite controlar exactamente qué sabe cada agente y evitar mezcla de fuentes.


In [ ]:
# ==============================================================================
# PYDANTIC MODELS + FUNCTION TOOLS
# ==============================================================================


# --- Modelos de datos ---

class EvidenciaLiteraria(BaseModel):
    """Evidencia textual recuperada de los corpus."""

    fuente: str = Field(description="'quijote' o 'critica'")
    fragmento: str = Field(description="Texto del fragmento recuperado")
    relevancia: float = Field(description="Score de similitud (0-1)")


class SynthesisOutput(BaseModel):
    """Output final del sistema A2A: análisis literario completo."""

    tesis_principal: str = Field(description="Argumento central del análisis")
    evidencia_textual: list[str] = Field(
        description="Citas o referencias del texto primario"
    )
    conexiones_literarias: list[str] = Field(
        description="Conexiones con otros autores/tradiciones (de la crítica)"
    )
    conclusion: str = Field(description="Síntesis final del análisis")
    nivel_confianza: str = Field(
        description="'alto', 'medio' o 'bajo' según la coherencia de la evidencia"
    )


# --- Function Tools ---

@function_tool
def buscar_texto_quijote(query: str, k: int = 3) -> str:
    """Busca los fragmentos más relevantes del texto original del Quijote.
    Retorna JSON con lista de {chunk_id, score, texto}."""
    embedding = embeddings_model.embed_query(query)
    results = col_quijote.query(
        query_embeddings=[embedding],
        n_results=min(k, col_quijote.count()),
        include=["documents", "distances"],
    )
    chunks = [
        {
            "chunk_id": results["ids"][0][i],
            "score": round(1 - results["distances"][0][i], 3),
            "texto": results["documents"][0][i][:400],
        }
        for i in range(len(results["ids"][0]))
    ]
    return json.dumps(chunks, ensure_ascii=False)


@function_tool
def buscar_critica_literaria(query: str, k: int = 3) -> str:
    """Busca los documentos de crítica literaria más relevantes para la query.
    Retorna JSON con lista de {doc_id, titulo, score, texto}."""
    embedding = embeddings_model.embed_query(query)
    results = col_critica.query(
        query_embeddings=[embedding],
        n_results=min(k, col_critica.count()),
        include=["documents", "distances", "metadatas"],
    )
    docs = [
        {
            "doc_id": results["ids"][0][i],
            "titulo": results["metadatas"][0][i].get("titulo", ""),
            "score": round(1 - results["distances"][0][i], 3),
            "texto": results["documents"][0][i][:500],
        }
        for i in range(len(results["ids"][0]))
    ]
    return json.dumps(docs, ensure_ascii=False)


@function_tool
def identificar_motivos_recurrentes(texto: str) -> str:
    """Analiza un texto literario e identifica los motivos y temas recurrentes.
    Retorna JSON con lista de {motivo, descripcion}."""
    prompt = (
        f"Analiza el siguiente texto literario e identifica los 3-5 motivos o temas más importantes.\n"
        f"Para cada motivo, proporciona nombre y descripción breve.\n"
        f"Responde en JSON con formato: [{{\"motivo\": ..., \"descripcion\": ...}}]\n\n"
        f"Texto:\n{texto[:2000]}"
    )
    resp = llm.invoke(prompt)
    return resp.content


@function_tool
def evaluar_coherencia_tematica(tesis: str, evidencias: str) -> str:
    """Evalua si las evidencias literarias respaldan coherentemente la tesis propuesta.
    Retorna JSON con: coherente (bool), nivel (alto/medio/bajo), observaciones."""
    prompt = (
        f"Evalúa si las siguientes evidencias respaldan coherentemente la tesis.\n\n"
        f"Tesis: {tesis}\n\n"
        f"Evidencias:\n{evidencias[:2000]}\n\n"
        'Responde en JSON: {"coherente": bool, "nivel": "alto|medio|bajo", "observaciones": str}'
    )
    resp = llm.invoke(prompt)
    return resp.content


print("Pydantic models y function tools definidos:")
print("  Modelos: EvidenciaLiteraria, SynthesisOutput")
print("  Tools:   buscar_texto_quijote, buscar_critica_literaria,")
print("           identificar_motivos_recurrentes, evaluar_coherencia_tematica")

## Arquitectura A2A: Handoffs vs LangGraph

### ¿Qué es un handoff?

Un **handoff** en el OpenAI Agents SDK transfiere el control completo de un agente a otro.
No es una llamada a función que retorna al agente original — es una **transferencia de control**.

```
Tool call:   Agent A → tool() → resultado → Agent A (continúa)
Handoff:     Agent A → handoff(Agent B) → Agent B (Agent A termina)
```

### Por qué 4 agentes especializados

| Agente | Rol | Por qué existe |
|--------|-----|---------------|
| **OrchestradorLiterario** | Entry point, coordina | Decide qué análisis hacer sin contaminar los datos |
| **ArchivistaLiterario** | Recuperación dual | Especialista en búsqueda: texto primario + crítica |
| **AnalistaTemasLiterarios** | Análisis de patrones | Especialista en identificar motivos, conexiones intertextuales |
| **SintetizadorLiterario** | Síntesis final | Especialista en producir análisis coherente con output tipado |

### Cadena de handoffs

```
OrchestradorLiterario (entry point)
    └─handoff→ ArchivistaLiterario
                   tools: [buscar_texto_quijote, buscar_critica_literaria]
                   └─handoff→ AnalistaTemasLiterarios
                                  tools: [identificar_motivos_recurrentes,
                                          buscar_texto_quijote]
                                  └─handoff→ SintetizadorLiterario
                                                 tools: [evaluar_coherencia_tematica]
                                                 output_type: SynthesisOutput
```

### Diseño correcto: orden inverso de creación

Para que cada agente tenga sus handoffs definidos al crearse, se crean en **orden inverso**:  
`SintetizadorLiterario` → `AnalistaTemasLiterarios` → `ArchivistaLiterario` → `OrchestradorLiterario`


In [ ]:
# ==============================================================================
# DEFINIR LOS 4 AGENTES (orden inverso: sintetizador -> analista -> archivista -> orquestador)
# ==============================================================================

# --- Agente 4: SintetizadorLiterario ---
# Último en la cadena. Recibe análisis del AnalistaTemasLiterarios y produce el output final.
sintetizador_literario = Agent(
    name="SintetizadorLiterario",
    instructions="""
Eres un sintetizador literario especializado en producir análisis cohesivos.

Recibirás un análisis previo del AnalistaTemasLiterarios con motivos identificados
y evidencias textuales. Tu tarea:

1. Llama a evaluar_coherencia_tematica() con la tesis principal y las evidencias compiladas.
2. Basándote en la evaluación, produce el output final estructurado SynthesisOutput con:
   - tesis_principal: el argumento central del análisis
   - evidencia_textual: lista de citas concretas del Quijote
   - conexiones_literarias: lista de conexiones con García Márquez, Borges, Kafka, etc.
   - conclusion: síntesis en 2-3 oraciones
   - nivel_confianza: 'alto' si coherente=true, 'medio' si hay algunas inconsistencias, 'bajo' si no

Sé preciso y académico. Usa la evidencia proporcionada.
""",
    tools=[evaluar_coherencia_tematica],
    output_type=SynthesisOutput,
    model="gpt-4o-mini",
)

# --- Agente 3: AnalistaTemasLiterarios ---
# Recibe evidencia del Archivista, identifica motivos, pasa al Sintetizador.
analista_temas = Agent(
    name="AnalistaTemasLiterarios",
    instructions="""
Eres un analista de temas literarios especializado en el Quijote y sus influencias.

Recibirás evidencia textual recopilada por el ArchivistaLiterario. Tu tarea:

1. Llama a identificar_motivos_recurrentes() con el texto de evidencia compilado.
2. Si necesitas más evidencia del Quijote para un motivo específico, usa buscar_texto_quijote().
3. Compila un análisis completo con: motivos identificados, evidencias textuales y
   conexiones con la crítica literaria recibida.
4. Pasa todo al SintetizadorLiterario via handoff para que produzca el análisis final.

IMPORTANTE: Tu mensaje al hacer handoff debe incluir todos los motivos, evidencias
y conexiones que encontraste.
""",
    tools=[identificar_motivos_recurrentes, buscar_texto_quijote],
    handoffs=[handoff(sintetizador_literario)],
    model="gpt-4o-mini",
)

# --- Agente 2: ArchivistaLiterario ---
# Entry point efectivo del análisis. Recupera de ambos corpus.
archivista_literario = Agent(
    name="ArchivistaLiterario",
    instructions="""
Eres un archivista literario especializado en recuperar evidencia textual y crítica.

Tu tarea:
1. Busca en el texto del Quijote con buscar_texto_quijote() usando la pregunta del usuario.
2. Busca en la crítica literaria con buscar_critica_literaria() para encontrar
   análisis académicos relevantes (conexiones con Borges, García Márquez, metaficción, etc.).
3. Compila toda la evidencia recuperada de ambos corpus.
4. Pasa la evidencia compilada al AnalistaTemasLiterarios via handoff.

IMPORTANTE: Tu mensaje al hacer handoff debe incluir:
- Los fragmentos del Quijote encontrados (con sus scores de relevancia)
- Los documentos de crítica literaria encontrados (con sus títulos)
- La pregunta original del usuario
""",
    tools=[buscar_texto_quijote, buscar_critica_literaria],
    handoffs=[handoff(analista_temas)],
    model="gpt-4o-mini",
)

# --- Agente 1: OrchestradorLiterario ---
# Entry point del sistema. Solo coordina, delega al Archivista.
orquestador_literario = Agent(
    name="OrchestradorLiterario",
    instructions="""
Eres el orquestador de un sistema de análisis literario especializado en el Quijote.

Tu rol es ÚNICAMENTE coordinar: no tienes tools de retrieval ni análisis.

Cuando recibas una pregunta de análisis literario:
1. Reformula la pregunta para maximizar la recuperación de evidencia relevante.
2. Delega inmediatamente al ArchivistaLiterario via handoff con la pregunta reformulada.

El ArchivistaLiterario se encargará de recuperar evidencia, el AnalistaTemasLiterarios
de identificar patrones, y el SintetizadorLiterario de producir el análisis final.

NO intentes hacer el análisis tú mismo. Tu valor está en la coordinación limpia.
""",
    handoffs=[handoff(archivista_literario)],
    model="gpt-4o-mini",
)

print("4 agentes definidos en orden inverso (sin mutación post-creación):")
print(f"  1. {orquestador_literario.name}: handoffs -> [{archivista_literario.name}]")
print(f"  2. {archivista_literario.name}: tools=[buscar_quijote, buscar_critica], handoffs -> [{analista_temas.name}]")
print(f"  3. {analista_temas.name}: tools=[identificar_motivos, buscar_quijote], handoffs -> [{sintetizador_literario.name}]")
print(f"  4. {sintetizador_literario.name}: tools=[evaluar_coherencia], output_type=SynthesisOutput")

In [ ]:
# ==============================================================================
# VISUALIZAR ARQUITECTURA A2A
# ==============================================================================

print("=" * 70)
print("ARQUITECTURA AGENT2AGENT: ANÁLISIS LITERARIO CERVANTINO")
print("=" * 70)
print()
print("  Usuario")
print("    │")
print("    ▼")
print("  ┌────────────────────────────────────────────────────────┐")
print("  │ OrchestradorLiterario (entry point)                    │")
print("  │   - Sin tools de retrieval                             │")
print("  │   - Reformula la query y delega                        │")
print("  └────────────────────────┬───────────────────────────────┘")
print("              handoff()   │")
print("                          ▼")
print("  ┌────────────────────────────────────────────────────────┐")
print("  │ ArchivistaLiterario                                    │")
print("  │   tools: [buscar_texto_quijote,                        │")
print("  │           buscar_critica_literaria]                    │")
print("  │   Recupera de 2 corpus independientes                  │")
print("  └────────────────────────┬───────────────────────────────┘")
print("              handoff()   │")
print("                          ▼")
print("  ┌────────────────────────────────────────────────────────┐")
print("  │ AnalistaTemasLiterarios                                │")
print("  │   tools: [identificar_motivos_recurrentes,             │")
print("  │           buscar_texto_quijote]                        │")
print("  │   Identifica patrones e intertextualidad               │")
print("  └────────────────────────┬───────────────────────────────┘")
print("              handoff()   │")
print("                          ▼")
print("  ┌────────────────────────────────────────────────────────┐")
print("  │ SintetizadorLiterario                                  │")
print("  │   tools: [evaluar_coherencia_tematica]                 │")
print("  │   output_type: SynthesisOutput (Pydantic)              │")
print("  └────────────────────────┬───────────────────────────────┘")
print("                          │")
print("                          ▼")
print("  SynthesisOutput(tesis, evidencias, conexiones, conclusion)")
print()
print("=" * 70)
print("DIFERENCIA CLAVE:")
print("  Tool call:  Agente A llama tool() -> resultado -> Agente A continúa")
print("  Handoff:    Agente A llama handoff(B) -> Agente B toma control")
print("              (Agente A NO continúa — transfiere el control)")
print("=" * 70)

## Cómo ejecutar y qué esperar en el trace

Al ejecutar la celda siguiente, el sistema A2A realizará:

1. **OrchestradorLiterario** recibe la query → handoff a Archivista
2. **ArchivistaLiterario** llama `buscar_texto_quijote()` y `buscar_critica_literaria()` → handoff a Analista
3. **AnalistaTemasLiterarios** llama `identificar_motivos_recurrentes()` → handoff a Sintetizador
4. **SintetizadorLiterario** llama `evaluar_coherencia_tematica()` → retorna `SynthesisOutput`

**Latencia esperada:** 15-40 segundos (4 agentes × múltiples llamadas LLM)

**Traza en OpenAI:** Disponible en https://platform.openai.com/traces (si tienes acceso)

> **Nota:** La latencia mayor es una desventaja real del patrón A2A. Se justifica cuando la calidad y profundidad del análisis son prioritarias sobre la velocidad.


In [ ]:
# ==============================================================================
# TEST: Ejecutar el sistema A2A con la query de análisis literario
# ==============================================================================

TEST_QUERY = (
    "¿Cómo se manifiesta el idealismo vs realidad en el Quijote "
    "y qué influencia tuvo en la literatura hispanoamericana?"
)


async def run_literary_analysis(query: str) -> SynthesisOutput:
    """Ejecuta el sistema A2A completo y retorna el SynthesisOutput."""
    with trace("literary-a2a-analysis"):
        result = await Runner.run(orquestador_literario, query)
    return result.final_output


print(f"Ejecutando sistema A2A con query:")
print(f"  '{TEST_QUERY}'")
print()
print("Cadena de agentes: Orquestador → Archivista → Analista → Sintetizador")
print("Esto puede tomar 20-40 segundos...")
print()

t0 = time.perf_counter()
loop = asyncio.get_event_loop()
literary_result = loop.run_until_complete(run_literary_analysis(TEST_QUERY))
latency = time.perf_counter() - t0

print(f"Completado en {latency:.1f}s")

In [ ]:
# ==============================================================================
# IMPRIMIR RESULTADO COMPLETO
# ==============================================================================

print("=" * 70)
print("RESULTADO DEL SISTEMA A2A: ANÁLISIS LITERARIO")
print("=" * 70)
print()
print("TESIS PRINCIPAL:")
print(f"  {literary_result.tesis_principal}")
print()
print("EVIDENCIA TEXTUAL DEL QUIJOTE:")
for i, ev in enumerate(literary_result.evidencia_textual, 1):
    print(f"  [{i}] {ev}")
print()
print("CONEXIONES LITERARIAS (influencia en s. XX):")
for i, cx in enumerate(literary_result.conexiones_literarias, 1):
    print(f"  [{i}] {cx}")
print()
print("CONCLUSIÓN:")
print(f"  {literary_result.conclusion}")
print()
print(f"NIVEL DE CONFIANZA: {literary_result.nivel_confianza.upper()}")
print("=" * 70)

## Evaluación: Agente RAG Simple vs Sistema A2A

Para medir objetivamente el valor del patrón A2A, comparamos contra un baseline:
un `AgenteRAGSimple` que tiene acceso a **las mismas tools** pero sin la cadena de handoffs.

### Métricas

| Métrica | Descripción | Por qué importa |
|---------|-------------|----------------|
| **Keyword recall** | % de keywords de ground truth en la respuesta | Precisión factual básica |
| **Cobertura temática** | % de los 5 temas de crítica mencionados | Profundidad del análisis |
| **Latencia** | Segundos totales | Costo de calidad vs velocidad |


In [ ]:
# ==============================================================================
# BASELINE: AgenteRAGSimple (mismas tools, sin handoffs)
# ==============================================================================

agente_rag_simple = Agent(
    name="AgenteRAGSimple",
    instructions="""
Eres un agente de análisis literario que responde preguntas sobre el Quijote.

Para responder:
1. Usa buscar_texto_quijote() para encontrar fragmentos relevantes.
2. Usa buscar_critica_literaria() para encontrar análisis académicos.
3. Produce una respuesta completa que integre ambas fuentes.

Responde directamente sin delegar a otros agentes.
""",
    tools=[buscar_texto_quijote, buscar_critica_literaria],
    model="gpt-4o-mini",
)

print("Baseline definido: AgenteRAGSimple")
print("  - Mismas tools que el sistema A2A")
print("  - Sin handoffs: un solo agente hace todo")
print("  - Comparacion justa: mismo modelo, mismas tools")

In [ ]:
# ==============================================================================
# QUERIES DE EVALUACION + GROUND TRUTH
# ==============================================================================

EVAL_QUERIES = [
    {
        "query": "¿Qué representa la locura de Don Quijote filosóficamente?",
        "keywords": ["epistémica", "epistemología", "percepción", "realidad", "Descartes", "constructivismo"],
        "temas_critica": ["locura", "epistem", "Descartes"],
    },
    {
        "query": "¿Cómo influenció el Quijote a Borges y García Márquez?",
        "keywords": ["Borges", "García Márquez", "metaficción", "realismo mágico", "hispanoamericano"],
        "temas_critica": ["Borges", "García Márquez", "modernismo", "metaficción"],
    },
    {
        "query": "¿Cuál es la relación filosófica entre Don Quijote y Sancho Panza?",
        "keywords": ["empirismo", "idealismo", "dialéctica", "racionalismo", "epistémico"],
        "temas_critica": ["Sancho", "empirismo", "dialéctica"],
    },
    {
        "query": "¿Por qué el Quijote es considerado la primera novela moderna occidental?",
        "keywords": ["metaficción", "autoconciencia", "narrador", "Cide Hamete", "posmoderna"],
        "temas_critica": ["metaficción", "posmoderna", "narrador"],
    },
]

TEMAS_CRITICA_TOTAL = [
    "locura epistémica",
    "idealismo hispanoamericano",
    "Sancho racionalismo",
    "metaficción occidental",
    "Rocinante simbolismo",
]

print(f"Queries de evaluación: {len(EVAL_QUERIES)}")
print(f"Temas de crítica (ground truth): {len(TEMAS_CRITICA_TOTAL)}")
for i, q in enumerate(EVAL_QUERIES, 1):
    print(f"  Q{i}: {q['query'][:60]}...")

In [ ]:
# ==============================================================================
# EVALUACION COMPARATIVA: Simple RAG vs A2A
# ==============================================================================


def keyword_recall(response: str, keywords: list[str]) -> float:
    """Calcula qué fracción de las keywords aparecen en la respuesta (case-insensitive)."""
    resp_lower = response.lower()
    hits = sum(1 for kw in keywords if kw.lower() in resp_lower)
    return hits / len(keywords) if keywords else 0.0


def cobertura_temas(response: str, temas: list[str]) -> float:
    """Calcula qué fracción de los temas de crítica aparecen en la respuesta."""
    resp_lower = response.lower()
    hits = sum(1 for tema in temas if any(w.lower() in resp_lower for w in tema.split()))
    return hits / len(temas) if temas else 0.0


async def evaluar_sistema(queries: list[dict]):
    """Evalua Simple RAG vs A2A en las queries dadas."""
    resultados = []
    for i, q_data in enumerate(queries, 1):
        query = q_data["query"]
        keywords = q_data["keywords"]
        temas = q_data["temas_critica"]
        print(f"  Evaluando Q{i}: {query[:50]}...")

        # --- Simple RAG ---
        t0 = time.perf_counter()
        result_simple = await Runner.run(agente_rag_simple, query)
        latency_simple = time.perf_counter() - t0
        resp_simple = str(result_simple.final_output)

        # --- A2A ---
        t0 = time.perf_counter()
        with trace(f"eval-a2a-q{i}"):
            result_a2a = await Runner.run(orquestador_literario, query)
        latency_a2a = time.perf_counter() - t0
        output_a2a: SynthesisOutput = result_a2a.final_output
        resp_a2a = (
            f"{output_a2a.tesis_principal} "
            f"{' '.join(output_a2a.evidencia_textual)} "
            f"{' '.join(output_a2a.conexiones_literarias)} "
            f"{output_a2a.conclusion}"
            if isinstance(output_a2a, SynthesisOutput)
            else str(output_a2a)
        )

        resultados.append({
            "query": query[:45] + "...",
            "simple_recall": keyword_recall(resp_simple, keywords),
            "a2a_recall": keyword_recall(resp_a2a, keywords),
            "simple_cobertura": cobertura_temas(resp_simple, temas),
            "a2a_cobertura": cobertura_temas(resp_a2a, temas),
            "simple_latencia": latency_simple,
            "a2a_latencia": latency_a2a,
        })
    return resultados


print("Iniciando evaluacion comparativa...")
print("(8 llamadas al LLM: 4 queries × 2 sistemas — puede tomar 2-3 minutos)")
print()

loop = asyncio.get_event_loop()
eval_results = loop.run_until_complete(evaluar_sistema(EVAL_QUERIES))

# Mostrar tabla
df_eval = pd.DataFrame(eval_results)
print()
print("=" * 80)
print("RESULTADOS DE EVALUACIÓN")
print("=" * 80)
display_cols = ["query", "simple_recall", "a2a_recall", "simple_cobertura",
                "a2a_cobertura", "simple_latencia", "a2a_latencia"]
print(
    df_eval[display_cols].to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)
print()
print("Promedios:")
print(f"  Keyword Recall  — Simple: {df_eval['simple_recall'].mean():.2f} | A2A: {df_eval['a2a_recall'].mean():.2f}")
print(f"  Cobertura temas — Simple: {df_eval['simple_cobertura'].mean():.2f} | A2A: {df_eval['a2a_cobertura'].mean():.2f}")
print(f"  Latencia (s)    — Simple: {df_eval['simple_latencia'].mean():.1f} | A2A: {df_eval['a2a_latencia'].mean():.1f}")

In [ ]:
# ==============================================================================
# VISUALIZACION: Bar chart precision + latencia
# ==============================================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Evaluación: Simple RAG vs Sistema A2A", fontsize=14, fontweight="bold")

queries_short = [f"Q{i+1}" for i in range(len(df_eval))]
x = np.arange(len(queries_short))
width = 0.35

# --- Plot 1: Keyword Recall ---
ax1 = axes[0]
bars1 = ax1.bar(x - width/2, df_eval["simple_recall"], width, label="Simple RAG", color="#4C72B0", alpha=0.85)
bars2 = ax1.bar(x + width/2, df_eval["a2a_recall"], width, label="A2A", color="#DD8452", alpha=0.85)
ax1.set_xlabel("Query")
ax1.set_ylabel("Keyword Recall")
ax1.set_title("Keyword Recall")
ax1.set_xticks(x)
ax1.set_xticklabels(queries_short)
ax1.set_ylim(0, 1.1)
ax1.legend()
ax1.axhline(y=df_eval["simple_recall"].mean(), color="#4C72B0", linestyle="--", alpha=0.5)
ax1.axhline(y=df_eval["a2a_recall"].mean(), color="#DD8452", linestyle="--", alpha=0.5)

# --- Plot 2: Cobertura Tematica ---
ax2 = axes[1]
ax2.bar(x - width/2, df_eval["simple_cobertura"], width, label="Simple RAG", color="#4C72B0", alpha=0.85)
ax2.bar(x + width/2, df_eval["a2a_cobertura"], width, label="A2A", color="#DD8452", alpha=0.85)
ax2.set_xlabel("Query")
ax2.set_ylabel("Cobertura de Temas")
ax2.set_title("Cobertura Temática")
ax2.set_xticks(x)
ax2.set_xticklabels(queries_short)
ax2.set_ylim(0, 1.1)
ax2.legend()

# --- Plot 3: Latencia ---
ax3 = axes[2]
ax3.bar(x - width/2, df_eval["simple_latencia"], width, label="Simple RAG", color="#4C72B0", alpha=0.85)
ax3.bar(x + width/2, df_eval["a2a_latencia"], width, label="A2A", color="#DD8452", alpha=0.85)
ax3.set_xlabel("Query")
ax3.set_ylabel("Latencia (s)")
ax3.set_title("Latencia Total")
ax3.set_xticks(x)
ax3.set_xticklabels(queries_short)
ax3.legend()

plt.tight_layout()
plt.savefig("/tmp/a2a_evaluacion.png", dpi=120, bbox_inches="tight")
plt.show()
print("Gráfico guardado en /tmp/a2a_evaluacion.png")

## Análisis Crítico: ¿Cuándo vale la pena A2A?

### Cuándo A2A supera a Simple RAG

| Condición | Por qué A2A gana |
|-----------|------------------|
| **Múltiples dominios** | Cada agente es especialista en su corpus; no mezcla fuentes |
| **Análisis profundo** | La cadena acumula contexto progresivamente |
| **Output tipado complejo** | El `SynthesisOutput` garantiza estructura consistente |
| **Trazabilidad** | Los handoffs son visibles en OpenAI Traces |

### Cuándo Simple RAG es mejor

| Condición | Por qué Simple RAG gana |
|-----------|-------------------------|
| **Latencia crítica** | A2A es 3-5x más lento (múltiples llamadas LLM) |
| **Queries simples** | Un agente generalista es suficiente para preguntas directas |
| **Presupuesto limitado** | Más tokens = más costo |
| **Prototipado** | Menor complejidad de configuración |

### Limitaciones del diseño actual

1. **Contexto no garantizado**: Los agentes pasan información via mensajes de texto — si el Archivista es verbose, puede truncar evidencia importante.
2. **Sin estado persistente**: Cada query parte de cero. No hay memoria entre ejecuciones.
3. **Self-grading en SintetizadorLiterario**: El mismo LLM evalúa su propio output.
4. **Corpus sintético**: La crítica literaria es fabricada; en producción requiere documentos reales.

### Roadmap a producción

1. **Memoria compartida**: Usar `ThreadLocalStorage` del SDK para pasar evidencia estructurada entre agentes
2. **Guardrails de entrada**: Validar que la query es de dominio literario antes de ejecutar la cadena
3. **Evaluador externo**: Reemplazar `evaluar_coherencia_tematica` con un agente evaluador independiente
4. **Corpus real**: Integrar PDFs de JSTOR, Google Scholar via `buscar_critica_literaria` real


## Checklist de Consolidación

Antes de continuar al siguiente módulo, asegúrate de poder responder:

1. **¿Cuál es la diferencia fundamental entre `handoff()` y un `function_tool`?**
   - *Pista: ¿quién continúa ejecutando después?*

2. **¿Por qué los agentes se crean en orden inverso en este notebook?**
   - *Pista: relación entre dependencias y orden de inicialización en Python*

3. **¿Qué ventaja ofrece `output_type=SynthesisOutput` en el SintetizadorLiterario?**
   - *Pista: Pydantic + structured output del SDK*

4. **¿Por qué el OrchestradorLiterario no tiene tools de retrieval?**
   - *Pista: separación de responsabilidades en sistemas multi-agente*

5. **¿En qué escenario real usarías A2A vs Single Agent RAG?**
   - *Pista: considera latencia, calidad, costo y complejidad del dominio*

---

### Referencias

- OpenAI Agents SDK: https://openai.github.io/openai-agents-python/
- Chip Huyen (2024). *AI Engineering: Building Applications with Foundation Models*. O'Reilly.
- Asai, A., et al. (2023). *Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection*. arXiv:2310.11511.
- Borges, J.L. (1944). *Pierre Menard, autor del Quijote*. En *Ficciones*.
